# Factor Analysis — Latent Factors and Specific Noise

## 1. Definition and intuition
Factor Analysis (FA) models observed features as combinations of a smaller set of latent factors plus feature-specific noise. Unlike PCA, FA explicitly models noise variance separately for each feature and is a probabilistic generative model.

## 2. Generative model
$$
x = \mu + Wz + \epsilon,\quad z\sim\mathcal N(0,I),\quad
\epsilon\sim\mathcal N(0,\Psi),
$$
where $W\in\mathbb R^{d\times q}$ is the loading matrix and $\Psi$ is diagonal. Thus $x\sim\mathcal N(\mu,WW^T+\Psi)$. The latent factors explain shared covariance; diagonal $\Psi$ captures feature-specific variance.

## 3. EM intuition
The E-step estimates the posterior moments of latent z given x. The M-step updates loadings and noise variances to improve the likelihood. Factor Analysis has rotational non-identifiability: multiple loading matrices can describe the same covariance, so individual loadings require care in interpretation.

## 4. Assumptions and failure modes
The standard model is linear-Gaussian with independent feature-specific residual noise. It can fail for nonlinear dependencies, heavy-tailed data or correlated residual noise. Scaling and number of factors affect interpretation.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import FactorAnalysis
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

SEED = 42
rng = np.random.default_rng(SEED)
n, d, q_true = 800, 10, 3
Z_true = rng.normal(size=(n, q_true))
W_true = rng.normal(size=(d, q_true))
noise_scale = rng.uniform(0.25, 0.8, size=d)
X = Z_true @ W_true.T + rng.normal(size=(n,d))*noise_scale + rng.normal(size=d)
X_train, X_test = train_test_split(X, test_size=0.25, random_state=SEED)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
print("Train shape:", X_train.shape)


Train shape: (600, 10)


## 5. Factor Analysis via EM from scratch

We use centered data and diagonal residual covariance. With $\Psi$ diagonal, posterior covariance is $V=(I+W^T\Psi^{-1}W)^{-1}$, and posterior means are $E[z|x]=x\Psi^{-1}WV$. The M-step updates W and the diagonal of $\Psi$. A small floor prevents zero noise variance.

This is an educational EM implementation; production implementations include additional convergence, numerical-stability and initialization strategies.

In [2]:
def fit_factor_analysis_em(X, n_factors, max_iter=200, tol=1e-5, noise_floor=1e-5, random_state=42):
    X = np.asarray(X, dtype=float)
    if X.ndim != 2 or len(X) < 2 or not np.isfinite(X).all():
        raise ValueError("X must be a finite 2D array with at least two rows")
    n, d = X.shape
    if not 1 <= n_factors <= min(n, d) or max_iter < 1 or tol < 0 or noise_floor <= 0:
        raise ValueError("Invalid factor-analysis parameters")
    mean = X.mean(axis=0)
    Xc = X - mean
    rng = np.random.default_rng(random_state)
    _, s, Vt = np.linalg.svd(Xc, full_matrices=False)
    W = Vt[:n_factors].T * (s[:n_factors] / np.sqrt(n)) [None, :]
    Psi = np.maximum(Xc.var(axis=0) * 0.5, noise_floor)
    history = []
    for _ in range(max_iter):
        invPsi = 1.0 / np.maximum(Psi, noise_floor)
        M = np.eye(n_factors) + W.T @ (invPsi[:, None] * W)
        V = np.linalg.inv(M)
        Ez = (Xc * invPsi) @ W @ V
        Ezz_sum = n * V + Ez.T @ Ez
        W_new = (Xc.T @ Ez) @ np.linalg.inv(Ezz_sum)
        S = Xc.T @ Xc / n
        cross = (Xc.T @ Ez) / n
        Ezz_avg = Ezz_sum / n
        cov_resid = S - cross @ W_new.T - W_new @ cross.T + W_new @ Ezz_avg @ W_new.T
        Psi_new = np.maximum(np.diag(cov_resid), noise_floor)
        delta = np.linalg.norm(W_new-W) / (np.linalg.norm(W) + 1e-12) + np.linalg.norm(Psi_new-Psi)/(np.linalg.norm(Psi)+1e-12)
        W, Psi = W_new, Psi_new
        history.append(float(delta))
        if delta < tol:
            break
    return {"mean": mean, "loadings": W, "noise_variance": Psi, "convergence_history": history}

def transform_factor_analysis(X, model):
    Xc = np.asarray(X, dtype=float) - model["mean"]
    W, Psi = model["loadings"], model["noise_variance"]
    invPsi = 1.0 / Psi
    V = np.linalg.inv(np.eye(W.shape[1]) + W.T @ (invPsi[:, None]*W))
    return (Xc * invPsi) @ W @ V

fa_scratch = fit_factor_analysis_em(X_train, n_factors=3, random_state=SEED)
Z_scratch = transform_factor_analysis(X_test, fa_scratch)
X_recon_scratch = Z_scratch @ fa_scratch["loadings"].T + fa_scratch["mean"]
print("Scratch factors:", Z_scratch.shape)
print("Scratch reconstruction MSE:", mean_squared_error(X_test, X_recon_scratch))
print("Estimated noise variances:", np.round(fa_scratch["noise_variance"], 3))


Scratch factors: (200, 3)
Scratch reconstruction MSE: 0.057327947648856535
Estimated noise variances: [0.068 0.055 0.07  0.059 0.061 0.136 0.01  0.119 0.096 0.049]


## 6. Compare with scikit-learn

Factor loadings may differ due to rotation and local optima. Compare reconstruction and latent dimensions, not raw loading matrices.

In [3]:
fa = FactorAnalysis(n_components=3, max_iter=1000, tol=1e-4, random_state=SEED).fit(X_train)
Z_sk = fa.transform(X_test)
X_recon_sk = Z_sk @ fa.components_ + fa.mean_
print("sklearn reconstruction MSE:", mean_squared_error(X_test, X_recon_sk))
print("Scratch vs sklearn reconstruction MSE:", round(mean_squared_error(X_test, X_recon_scratch), 4),
      round(mean_squared_error(X_test, X_recon_sk), 4))
assert Z_scratch.shape == Z_sk.shape
assert np.isfinite(Z_scratch).all() and np.isfinite(fa_scratch["loadings"]).all()


sklearn reconstruction MSE: 0.0573024703760181
Scratch vs sklearn reconstruction MSE: 0.0573 0.0573


## 7. Hyperparameters, complexity and scale
- `n_components`: number of latent factors q.
- `max_iter`, `tol`: EM stopping criteria.
- Full covariance reasoning costs roughly $O(ndq + nq^2 + dq^2)$ per EM iteration; the exact complexity depends on algebraic ordering. Storage is $O(nd+dq+q^2)$.
- For large n and modest d, process batches for sufficient statistics when using a suitable streaming EM design. For large d, covariance calculations are the bottleneck.
- GPU matrix operations are possible, but iterative small matrix inversions can dominate when q is small; benchmark before porting.

## 8. Best practices and synthesis
FA is useful when a latent linear Gaussian structure and feature-specific noise are plausible. PCA maximizes sample variance in a deterministic subspace; FA explains covariance with a probabilistic latent-variable model. Use multiple starts and validate factor count and residual structure.

In [4]:
assert np.isfinite(fa_scratch["noise_variance"]).all()
assert np.all(fa_scratch["noise_variance"] > 0)
assert len(fa_scratch["convergence_history"]) >= 1
try:
    fit_factor_analysis_em(np.array([1,2,3]), 1)
except ValueError:
    pass
else:
    raise AssertionError("Expected invalid input shape to raise")
print("All Factor Analysis checks passed.")


All Factor Analysis checks passed.
